# Week 2 — exhaustive vs greedy, two problems

Open in VS Code. **Select Kernel** (Python 3) in the top right. Then **Run All** from the top.

Most cells use the standard library only. The genetic-algorithm section also needs `matplotlib` so it can show an animation of the tour improving.
Change parameters and re-run.

- Toy 1: 4 cities $A,B,C,D$ — list tours, nearest neighbor, 2-opt; then a **15-city** genetic algorithm with live tour redraw.
- Toy 2: interval scheduling — pass your own list of talks.


In [ ]:
import random

cities = ["A", "B", "C", "D"]
# Lecture matrix: NN from A and from B differ; neither is the shortest (12).
D = [
    #  A  B  C  D
    [0, 2, 6, 3],  # A
    [2, 0, 3, 1],  # B
    [6, 3, 0, 4],  # C
    [3, 1, 4, 0],  # D
]


def tour_length(tour, D):
    # tour is indices; add the hop home.
    s = 0
    seq = list(tour) + [tour[0]]
    for a, b in zip(seq, seq[1:]):
        s += D[a][b]
    return s


## How many tours?

Up to rotation and direction: $(n-1)!/2$.


In [ ]:
n = len(cities)
n_tours = 1
for k in range(1, n):
    n_tours *= k
n_tours //= 2
print("(n-1)!/2 =", n_tours)


## Three tours starting at $A$

Same cycle backwards, or rotated, is not a new tour.


In [ ]:
tours = [
    (0, 1, 2, 3),  # A B C D
    (0, 1, 3, 2),  # A B D C
    (0, 2, 1, 3),  # A C B D
]
for t in tours:
    names = " ".join(cities[i] for i in t)
    print(names, " length", tour_length(t, D))
print("shortest of these is 12")


## Nearest neighbor

Always the closest unused city, then home. Try start $0$ ($A$) and start $1$ ($B$).


In [ ]:
def nn_tour(D, start):
    n = len(D)
    used = {start}
    tour = [start]
    here = start
    while len(tour) < n:
        nxt = min((j for j in range(n) if j not in used), key=lambda j: D[here][j])
        tour.append(nxt)
        used.add(nxt)
        here = nxt
    return tuple(tour)


for start in (0, 1):
    t = nn_tour(D, start)
    path = " -> ".join(cities[i] for i in t) + " -> " + cities[t[0]]
    print(f"NN from {cities[start]}: {path}   length {tour_length(t, D)}")


## 2-opt

NN from $A$ is $A \to B \to D \to C$. Drop the diagonals $B$–$D$ and $C$–$A$, add $B$–$C$ and $D$–$A$. That is the square $A \to B \to C \to D$.


In [ ]:
nn_A = (0, 1, 3, 2)  # A B D C
opt = (0, 1, 2, 3)   # A B C D  after swapping BD+CA for BC+DA
print("NN from A", tour_length(nn_A, D))
print("after 2-opt", tour_length(opt, D))


## Genetic algorithm — more cities, watch the tour

The 4-city matrix above is for listing tours by hand. Here we use **15 cities in the plane** (Euclidean distances), so a random tour is messy and improvement is visible.

Change `POP_SIZE`, `GENERATIONS`, `MUTATION_RATE`, and re-run. After the run finishes, an **animation** of the best tour appears — press ▶ to play it.

Needs `matplotlib` (already working if you see plots).


In [ ]:
import math
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

# --- parameters you can vary ---
N_CITIES = 25
POP_SIZE = 100
GENERATIONS = 200
MUTATION_RATE = 0.25
SEED = 1
DRAW_EVERY = 2          # store a frame every k generations
INTERVAL_MS = 120       # playback speed (ms per frame)
# ------------------------------

random.seed(SEED)

# Cities as points in the unit square
pts = [(random.random(), random.random()) for _ in range(N_CITIES)]

def euclid(i, j):
    return math.hypot(pts[i][0] - pts[j][0], pts[i][1] - pts[j][1])

D_big = [[euclid(i, j) for j in range(N_CITIES)] for i in range(N_CITIES)]
n = N_CITIES


def order_crossover(p1, p2):
    lo, hi = sorted(random.sample(range(n), 2))
    child = [None] * n
    child[lo:hi] = p1[lo:hi]
    fill = [c for c in p2 if c not in child[lo:hi]]
    k = 0
    for i in range(n):
        if child[i] is None:
            child[i] = fill[k]
            k += 1
    return child


def mutate(tour):
    t = tour[:]
    i, j = random.sample(range(n), 2)
    t[i], t[j] = t[j], t[i]
    return t


def genetic_tsp_frames(D, pop_size=POP_SIZE, generations=GENERATIONS, mutation_rate=MUTATION_RATE):
    """Run GA; return list of (generation, best_tour, best_len, history_so_far) and NN length."""
    pop = [random.sample(range(n), n) for _ in range(pop_size)]
    best_hist = []
    frames = []
    nn = nn_tour(D, 0)
    nn_len = tour_length(nn, D)

    for g in range(generations):
        scored = sorted(pop, key=lambda t: tour_length(t, D))
        best_tour = scored[0]
        best_len = tour_length(best_tour, D)
        best_hist.append(best_len)

        if g % DRAW_EVERY == 0 or g == generations - 1:
            frames.append((g, best_tour[:], best_len, best_hist[:]))

        survivors = scored[: pop_size // 2]
        children = []
        while len(survivors) + len(children) < pop_size:
            p1, p2 = random.sample(survivors, 2)
            child = order_crossover(p1, p2)
            if random.random() < mutation_rate:
                child = mutate(child)
            children.append(child)
        pop = survivors + children

    return frames, nn_len


def animate_ga(frames, nn_len):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    def draw_frame(frame):
        g, tour, best_len, hist = frame
        axes[0].clear()
        axes[1].clear()

        xs = [pts[i][0] for i in tour] + [pts[tour[0]][0]]
        ys = [pts[i][1] for i in tour] + [pts[tour[0]][1]]
        axes[0].plot(xs, ys, "-", color="#2a5a8c", lw=1.8)
        axes[0].scatter([p[0] for p in pts], [p[1] for p in pts], c="#b56a3a", s=40, zorder=3)
        for i, (x, y) in enumerate(pts):
            axes[0].text(x + 0.01, y + 0.01, str(i), fontsize=8)
        axes[0].set_xlim(-0.05, 1.05)
        axes[0].set_ylim(-0.05, 1.05)
        axes[0].set_aspect("equal")
        axes[0].set_title(f"generation {g}: best = {best_len:.3f}  (NN = {nn_len:.3f})")
        axes[0].set_xticks([])
        axes[0].set_yticks([])

        axes[1].plot(hist, color="#b56a3a")
        axes[1].axhline(nn_len, color="#2a5a8c", linestyle="--", label=f"NN {nn_len:.3f}")
        axes[1].set_xlabel("generation")
        axes[1].set_ylabel("best length")
        axes[1].legend(loc="upper right")
        axes[1].set_title("fitness over time")
        fig.tight_layout()

    anim = animation.FuncAnimation(
        fig,
        lambda i: draw_frame(frames[i]),
        frames=len(frames),
        interval=INTERVAL_MS,
        blit=False,
        repeat=True,
    )
    plt.close(fig)
    display(HTML(anim.to_jshtml()))
    return anim


frames, nn_len = genetic_tsp_frames(D_big)
print(f"computed {len(frames)} frames · final best {frames[-1][2]:.3f} · NN {nn_len:.3f}")
print("Play the animation below (press ▶).")
_anim = animate_ga(frames, nn_len)
print("Try larger POP_SIZE / GENERATIONS, or N_CITIES = 25.")


## Interval scheduling — two strategies

One room. As many talks as possible; two talks cannot overlap. A talk may start when the previous one finishes.

Both rules below are implemented. Pass **any** list of `(name, start, finish)`.

- **Earliest finish:** always take the remaining talk that finishes first.
- **Shortest first:** always take the remaining shortest talk that still fits.


In [ ]:
def earliest_finish(talks):
    """talks: list of (name, start, finish). Returns chosen names."""
    chosen, t = [], float("-inf")
    for name, s, f in sorted(talks, key=lambda x: x[2]):
        if s >= t:
            chosen.append(name)
            t = f
    return chosen


def shortest_first(talks):
    """talks: list of (name, start, finish). Returns chosen names."""
    chosen, busy = [], []
    for name, s, f in sorted(talks, key=lambda x: x[2] - x[1]):
        if all(f <= a or s >= b for a, b in busy):
            chosen.append(name)
            busy.append((s, f))
    return chosen


def compare_rules(talks):
    ef = earliest_finish(talks)
    sf = shortest_first(talks)
    print("earliest finish", ef, " count", len(ef))
    print("shortest first ", sf, " count", len(sf))


# Lecture toy (hours from noon)
toy = [("A", 0, 4), ("B", 3, 5), ("C", 4, 8)]
print("=== lecture toy ===")
compare_rules(toy)


### Q3 instance — explore here

Same table as on the slides (clock times as hours). Run both strategies. Answers are at the **end of the slide deck**, not here.

| Talk | Start | Finish |
|------|-------|--------|
| $A$ | 09:00 | 13:00 |
| $B$ | 12:00 | 14:00 |
| $C$ | 13:00 | 17:00 |
| $D$ | 17:00 | 18:00 |


In [ ]:
# Q3 table — edit freely, then re-run
q3 = [
    ("A", 9, 13),
    ("B", 12, 14),
    ("C", 13, 17),
    ("D", 17, 18),
]

print("=== Q3 ===")
compare_rules(q3)

# Optional: how many subsets of 4 talks?
print("number of subsets 2^4 =", 2 ** len(q3))

# Your own instance:
# my_talks = [("X", 0, 2), ("Y", 1, 3), ("Z", 2, 5)]
# compare_rules(my_talks)


## Questions

**Q1.** How many distinct TSP tours on $n$ cities, and why is that not $n!$?

A1.

**Q2.** Nearest neighbor from two starts can disagree, and neither need be shortest. Say that on the toy.

A2.

**Q3.** Use the interval cells above. Write your answers here; check against the end of the slides when ready.

(a) Listing every subset: how many subsets of 4 talks? Does that method give a largest possible set?

A3a.

(b) Earliest finish — which talks?

A3b.

(c) Shortest first — which talks?

A3c.

(d) Which of (a)–(c) is guaranteed a largest set? Which can return too few?

A3d.
